In [1]:
import sys
from pathlib import Path

from runner.runners import ZSSRRunner
from data.preprocessing import ZSSRPreprocessing
from data.datasets import Urban100Dataset
from eval.pipeline import SRPipeline
from eval.utils import generate_comparison_plots, plot_layer_distributions

from ipywidgets import interact, IntSlider, fixed
import ipywidgets as widgets

import config

In [2]:
NUM_PATCHES = 64
BATCH_SIZE = 1
SCALE_FACTOR = 4
N_SCALE_FACTORS = 3
N_EPOCHS = 20

In [3]:
"""
runner = ZSSRRunner()
pipeline = SRPipeline(
    runner=runner,
    dataset_zip_path=config.URBAN100_ZIP,
    datasets_dir=config.DATASET_DIR,
    output_dir=config.ZSSR_OUTPUT_DIR / 'sigmoid' / config.URBAN100_NAME ,
    scale_factor=4.0
)

pipeline.run()
"""

"\nrunner = ZSSRRunner()\npipeline = SRPipeline(\n    runner=runner,\n    dataset_zip_path=config.URBAN100_ZIP,\n    datasets_dir=config.DATASET_DIR,\n    output_dir=config.ZSSR_OUTPUT_DIR / 'sigmoid' / config.URBAN100_NAME ,\n    scale_factor=4.0\n)\n\npipeline.run()\n"

In [ ]:
runner = ZSSRRunner()

strategy = ZSSRPreprocessing(num_patches=64) 
dataset = Urban100Dataset(root_dir=str('datasets/temp/army/'), scale_factor=SCALE_FACTOR, strategy=strategy)

_, h, w = strategy.base_img.shape
out_size = (int(h * SCALE_FACTOR), int(w * SCALE_FACTOR))

runner.train(dataset, out_size=out_size)

--- Training ZSSR for 50 epochs ---
--- Training ZSSR with s_i=1.5 ---
End of epoch 0, Loss: 0.162496
End of epoch 1, Loss: 0.213631
End of epoch 2, Loss: 0.151120
End of epoch 3, Loss: 0.149662
End of epoch 4, Loss: 0.149458
End of epoch 5, Loss: 0.148588
End of epoch 6, Loss: 0.145739
End of epoch 7, Loss: 0.140022
End of epoch 8, Loss: 0.140108
End of epoch 9, Loss: 0.138774
End of epoch 10, Loss: 0.134085
End of epoch 11, Loss: 0.121353
End of epoch 12, Loss: 0.102981
End of epoch 13, Loss: 0.097212
End of epoch 14, Loss: 0.080948
End of epoch 15, Loss: 0.061721
End of epoch 16, Loss: 0.080137
End of epoch 17, Loss: 0.058724
End of epoch 18, Loss: 0.059613
End of epoch 19, Loss: 0.055839
End of epoch 20, Loss: 0.052908
End of epoch 21, Loss: 0.040630
End of epoch 22, Loss: 0.036931
End of epoch 23, Loss: 0.031589
End of epoch 24, Loss: 0.052341
End of epoch 25, Loss: 0.070368
End of epoch 26, Loss: 0.054010
End of epoch 27, Loss: 0.035028
End of epoch 28, Loss: 0.036342
End of epoc

Visualization of Weights and Gradient update

In [ ]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

max_epochs = len(runner.layer_stats['weights']) - 1
slider = widgets.IntSlider(min=0, max=max_epochs, step=1, value=0, description='Epoch:')
out = widgets.Output()

def update_plot(change=None): 
    epoch = slider.value 
    
    with out:
        clear_output(wait=True)
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
        
        # Plot Weight Histogram
        weights = runner.layer_stats['weights'][epoch]
        ax1.hist(weights, bins=50, color='royalblue', alpha=0.7)
        ax1.set_title(f'Weight Distribution (Epoch {epoch})')
        ax1.set_xlabel('Weight Value')
        ax1.grid(axis='y', alpha=0.3)
        
        # Plot Gradient Histogram
        grads = runner.layer_stats['gradients'][epoch]
        ax2.hist(grads, bins=50, color='darkorange', alpha=0.7)
        ax2.set_title(f'Gradient Distribution (Epoch {epoch})')
        ax2.set_xlabel('Gradient Value')
        ax2.grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
        plt.show()

slider.observe(update_plot, names='value')

display(slider, out)

update_plot()

Metrics

In [ ]:
folder_dict = {
    config.BSD100_NAME: {
        "zssr": config.ZSSR_OUTPUT_DIR / config.BSD100_NAME,
        "sigmoid": config.ZSSR_OUTPUT_DIR / "sigmoid" / config.BSD100_NAME
    },
    config.URBAN100_NAME: {
        "zssr": config.ZSSR_OUTPUT_DIR / config.URBAN100_NAME,
        "sigmoid": config.ZSSR_OUTPUT_DIR / "sigmoid" / config.URBAN100_NAME
    }
}    
generate_comparison_plots(folder_dict)